# Image Classification Pattern

Follow an image batch from dataset contract through a small CNN, evaluation, and filename-keyed inference.

- **Study time:** 50-70 minutes
- **Prerequisites:** PyTorch DataLoader, convolution shapes, and the canonical training loop
- **Mode:** `optional`
- **Data policy:** no downloads or image files; deterministic circle/square tensors are generated in memory
- **Provenance:** consolidated from the legacy CIFAR case study and portal dataset-pattern notebooks

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

seed = 61
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)  # Align all three RNGs for this executable example.
# Keep model and batches on one selected device.
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print("Environment | selected device", device)

## 1. Dataset returns `(C, H, W)`, integer label, and stable identifier


In [ ]:
class ShapeDataset(Dataset):
    classes = ("circle", "square")

    def __init__(self, n_samples=600, image_size=20, seed=61):
        generator = torch.Generator().manual_seed(
            seed
        )  # Dataset-local noise stays reproducible and isolated.
        self.labels = torch.arange(n_samples) % 2  # Deterministic balanced class sequence.
        self.filenames = [f"shape_{index:04d}.png" for index in range(n_samples)]
        coordinates = torch.linspace(-1, 1, image_size)
        yy, xx = torch.meshgrid(
            coordinates, coordinates, indexing="ij"
        )  # Pixel-coordinate grids: (H, W).
        circle = ((xx**2 + yy**2) <= 0.48**2).float()
        square = ((xx.abs() <= 0.48) & (yy.abs() <= 0.48)).float()
        templates = torch.stack([circle, square])[
            :, None, :, :
        ]  # Class templates in (classes, C, H, W).
        noise = 0.12 * torch.randn(n_samples, 1, image_size, image_size, generator=generator)
        self.images = (templates[self.labels] + noise).clamp(
            0, 1
        )  # Index the template for each label, then perturb pixels.

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.images[index], self.labels[index], self.filenames[index]


dataset = ShapeDataset()
train_dataset, validation_dataset, test_dataset = random_split(
    dataset,
    [420, 90, 90],
    generator=torch.Generator().manual_seed(61),  # Reproducible, disjoint subsets.
)
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True
)  # Shuffle training examples only.
validation_loader = DataLoader(
    validation_dataset, batch_size=128, shuffle=False
)  # Stable evaluation order.
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
sample_X, sample_y, sample_names = next(iter(train_loader))

print("Dataset | batch NCHW shape", tuple(sample_X.shape))
print(
    "Dataset | batch dtype and min/max",
    (sample_X.dtype, sample_X.min().item(), sample_X.max().item()),
)
print("Dataset | first labels and identifiers", (sample_y[:5].tolist(), list(sample_names[:5])))

## 2. Small CNN produces one logit per class


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),  # (N, 1, 20, 20) -> (N, 8, 20, 20).
            nn.ReLU(),
            nn.MaxPool2d(2),  # Halve spatial dimensions: (N, 8, 10, 10).
            nn.Conv2d(8, 16, kernel_size=3, padding=1),  # Preserve 10x10, increase channels to 16.
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # Collapse any spatial size to one value per channel.
        )
        self.classifier = nn.Linear(16, n_classes)

    def forward(self, X):
        features = self.features(X).flatten(1)  # (N, 16, 1, 1) -> (N, 16); keep batch axis.
        return self.classifier(features)


model = SmallCNN().to(device)
with torch.no_grad():  # Shape probe only; no backward graph is needed.
    sample_logits = model(sample_X.to(device))
print("Model | input and logits shapes", (tuple(sample_X.shape), tuple(sample_logits.shape)))

## 3. Train/evaluate without augmenting validation


In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = (
        optimizer is not None
    )  # One loop; optimizer presence selects train versus evaluation.
    model.train(training)  # Switch mode-sensitive layers; gradient context is handled separately.
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    context = torch.enable_grad() if training else torch.no_grad()  # Build graphs only for updates.
    with context:
        for X_batch, y_batch, _ in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            if training:
                optimizer.zero_grad(
                    set_to_none=True
                )  # Clear prior batch gradients before backward.
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(
                X_batch
            )  # Sample-weighted total handles a short final batch.
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += len(X_batch)
    return total_loss / total_examples, total_correct / total_examples


loss_fn = nn.CrossEntropyLoss()  # Expects raw (N, classes) logits and integer class indices.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1, 7):
    train_loss, train_accuracy = run_epoch(model, train_loader, loss_fn, optimizer)
    validation_loss, validation_accuracy = run_epoch(model, validation_loader, loss_fn)
    print(
        f"Training | epoch={epoch:02d} train_loss={train_loss:.4f} "
        f"train_acc={train_accuracy:.3f} validation_loss={validation_loss:.4f} "
        f"validation_acc={validation_accuracy:.3f}"
    )

## 4. Inference remains keyed by filename


In [ ]:
model.eval()  # Select inference behavior; no_grad below separately disables graph construction.
rows = []
with torch.no_grad():
    for X_batch, y_batch, names in test_loader:
        probabilities = torch.softmax(
            model(X_batch.to(device)), dim=1
        ).cpu()  # Return probabilities to CPU for reporting.
        predictions = probabilities.argmax(dim=1)
        for name, target, prediction, confidence in zip(
            names,
            y_batch,
            predictions,
            probabilities.max(dim=1).values,
            strict=True,  # Fail loudly if identifiers, labels, and predictions lose alignment.
        ):
            rows.append(
                {
                    "filename": name,
                    "target": dataset.classes[target.item()],
                    "prediction": dataset.classes[prediction.item()],
                    "confidence": confidence.item(),
                }
            )

inference = pd.DataFrame(rows)
test_accuracy = (inference["target"] == inference["prediction"]).mean()
confusion = pd.crosstab(inference["target"], inference["prediction"]).reindex(
    index=dataset.classes,
    columns=dataset.classes,
    fill_value=0,  # Retain classes even if a row/column has no observations.
)  # Rows are actual classes; columns are predicted classes.
per_class_recall = pd.Series(
    np.diag(confusion) / confusion.sum(axis=1),  # Correct predictions / actual examples per class.
    index=dataset.classes,
    name="recall",
)

assert confusion.to_numpy().sum() == len(inference)
assert per_class_recall.between(0, 1).all()
print("Inference | first five filename-keyed predictions", inference.head())
print("Inference | test accuracy", test_accuracy)
print("Inference | confusion matrix", confusion)
print("Inference | per-class recall", per_class_recall.round(3))
print(
    "Image checks | status",
    "NCHW, logits, model modes, class-level metrics, and identifier mapping verified",
)